# Pipeline for counting EFMs and calculating their thermodynamic properties

This notebook builds a COBRApy model of biochemical reactions from a custom Excel spreadsheet. It (1) checks stoichiometric balance of every reaction using the KEGG REST API, (2) runs Flux Balance Analysis (FBA) to compute maximum theoretical product yields from glucose, and (3) exports the final model to JSON and SBML formats.

## User Inputs

Edit **only the cell below**, then run all cells from top to bottom.

At a few points in the notebook you will encounter a box like this:

<div style="border-left:5px solid #2196F3; background:rgba(33,150,243,0.08); padding:8px 14px; margin:8px 0; border-radius:3px;">
<b>&#9998; TO DO:</b> Example action required.
</div>

These mark steps where you need to **check the output and possibly edit your Excel file** before continuing. Re-run all cells after making any changes.

| Variable | What to change |
|---|---|
| `EXCEL_FILE` | Name of your Excel file (must be in the same folder as this notebook) |
| `MODEL_NAME` | Name used for the output `.json` and `.xml` files |
| `SUBSTRATES` | Exchange reaction ID(s) for compounds the pathway **consumes** |
| `PRODUCTS` | Exchange reaction IDs for **all** secreted products (including energy currency) |
| `CARBON_PRODUCTS` | Subset of `PRODUCTS` to check maximum yields for (usually excludes ATP) |
| `ENERGY_PRODUCT` | The ATP (or equivalent) exchange ID — used to detect thermodynamically infeasible cycles |
| `REV_ALLOWED` | Metabolites that can be both taken up **and** secreted freely |
| `PSEUDO_METS` | Currency metabolites whose transport is **not** counted as an enzymatic step |


In [62]:
# ============================================================
# USER INPUTS — Edit this cell, then run all cells
# ============================================================

# Excel file with your pathway (must be in the same folder as this notebook)
EXCEL_FILE = 'Example1_EMPglycolysis.xlsx'

# Name for your model and output files (no spaces)
MODEL_NAME = 'Example1_EMPglycolysis'

# Exchange reaction ID for the substrate (compound the pathway consumes)
SUBSTRATES = ['EXGLC']

# All exchange reaction IDs that can be secreted as products
PRODUCTS = ['EXATP', 'EXETOH', 'EXAC', 'EXFOR']

# Products to check maximum yield for (exclude energy currency such as ATP)
CARBON_PRODUCTS = ['EXETOH', 'EXAC', 'EXFOR']

# Exchange reaction ID for the energy currency product (used to detect infeasible cycles)
ENERGY_PRODUCT = 'EXATP'

# Cofactor/currency exchanges that can be freely taken up or secreted
REV_ALLOWED = ['EXADP', 'EXPi', 'EXH2O', 'EXH']

# Currency metabolites whose transport is NOT counted as an enzymatic step
PSEUDO_METS = {'H2O', 'H2Oex', 'H', 'Hex', 'ATP', 'ATPex', 'ADP', 'ADPex', 'Pi', 'Piex'}


In [63]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [64]:
# Change directory
import os
os.chdir('/content/drive/MyDrive/Colab Notebooks')
cwd=os.getcwd()
os.listdir(cwd)

['educational_paper_pipeline.ipynb',
 'multipleEFMs_example.ipynb',
 'EFMexamples_24.xlsx',
 'Example1_EMPglycolysis.xlsx',
 'EFMs8_example.ipynb',
 'EFMexamples_8.xlsx']

## Imports

If you are running in **Google Colab**, add a code cell at the top and run:

```python
!pip install cobra
!pip install efmtool
!pip install equilibrator-api
```

> **Note:** `equilibrator-api` downloads thermodynamic data on first use — this can take a few minutes depending on your internet connection.


In [65]:
import math
import re
import time
from collections import Counter
from functools import lru_cache
!pip install cobra
!pip install efmtool
!pip install equilibrator_api
import cobra
import efmtool
import numpy as np
import pandas as pd
import requests

In [66]:
# Try to only run this once (only when you need it), it takes a while (depending on your wifi)
from equilibrator_api import ComponentContribution, Q_
cc = ComponentContribution()

## Helper Functions

Two utility functions used throughout the notebook: `dict_to_kegg_reaction` converts a metabolite stoichiometry dictionary into a KEGG-formatted reaction string (needed for eQuilibrator thermodynamics lookups), and `clean_dataframe_whitespace` strips newline and tab characters from all string cells in a DataFrame to ensure consistent parsing.

In [67]:
# Helper function for eQuilibrator
def dict_to_kegg_reaction(met_dict):
    reactants = []
    products = []

    for met, coeff in met_dict.items():
        if coeff < 0:
            term = f"{-coeff:g} kegg:{met}" if abs(coeff) != 1 else f"kegg:{met}"
            reactants.append(term)
        elif coeff > 0:
            term = f"{coeff:g} kegg:{met}" if abs(coeff) != 1 else f"kegg:{met}"
            products.append(term)

    lhs = ' + '.join(reactants)
    rhs = ' + '.join(products)
    return f"{lhs} = {rhs}"

In [68]:
def clean_dataframe_whitespace(df, inplace=False):
    """
    Remove all newline '\n' and tab '\t' characters from every string cell
    in a pandas DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        The dataframe to clean.
    inplace : bool, optional
        If True, modify the dataframe in place. Default is False.

    Returns
    -------
    pandas.DataFrame
        The cleaned dataframe (unless inplace=True).
    """

    target = df if inplace else df.copy()

    target = target.applymap(
        lambda x: x.replace('\n', '').replace('\t', '') if isinstance(x, str) else x
    )

    return target

## Load Reaction and Metabolite Data

Read the metabolite table and reaction list from the Excel workbook (`Example1_EMPglycolysis.xlsx`). Whitespace is cleaned from both DataFrames before further processing.

In [69]:
df_metabolites = pd.read_excel(EXCEL_FILE, sheet_name='Metabolites')
df_reactions = pd.read_excel(EXCEL_FILE, sheet_name='Reactions')

df_metabolites = clean_dataframe_whitespace(df_metabolites)
df_reactions = clean_dataframe_whitespace(df_reactions)


/tmp/ipykernel_1344/3564247440.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  target = target.applymap(


## Stoichiometric Balance Checking — Function Definitions

Defines the full pipeline for checking whether each reaction is atom- and charge-balanced:

- **KEGG API fetching** (`get_compound_info`): retrieves molecular formula and charge for a KEGG compound ID, with caching.
- **Equation parsing** (`parse_equation`, `normalize_arrow`): splits a reaction string into substrate and product lists.
- **Formula parsing** (`parse_formula`): converts a KEGG formula string (e.g. `C6H12O6`) into an element count dictionary; handles hydrates and unusual characters.
- **ID mapping** (`map_custom_to_kegg_equation`): translates the custom abbreviations used in the Excel sheet to KEGG compound IDs.
- **Balance check** (`check_balance`): computes the atom and charge delta across a reaction; returns a summary of any imbalances.
- **Pipeline functions** (`analyze_custom_equations`, `analyze_stoichiometry_matrix`): orchestrate the above steps for a list of reactions and return a results DataFrame.

In [70]:

# -------- ELEMENT SYMBOL WHITELIST (up to Og) --------
ELEMENTS = {
    "H","He","Li","Be","B","C","N","O","F","Ne","Na","Mg","Al","Si","P","S","Cl","Ar",
    "K","Ca","Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn","Ga","Ge","As","Se","Br","Kr",
    "Rb","Sr","Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd","In","Sn","Sb","Te","I","Xe",
    "Cs","Ba","La","Ce","Pr","Nd","Pm","Sm","Eu","Gd","Tb","Dy","Ho","Er","Tm","Yb","Lu",
    "Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg","Tl","Pb","Bi","Po","At","Rn",
    "Fr","Ra","Ac","Th","Pa","U","Np","Pu","Am","Cm","Bk","Cf","Es","Fm","Md","No","Lr",
    "Rf","Db","Sg","Bh","Hs","Mt","Ds","Rg","Cn","Nh","Fl","Mc","Lv","Ts","Og"
}

# -------- KEGG API (robust parsing + caching) --------
@lru_cache(maxsize=10000)
def get_compound_info(cid):
    """
    Return (formula_str, charge_int) from KEGG REST for a Cxxxxx id.
    Robust to variable spacing; ignores non-matching lines.
    """
    url = f"http://rest.kegg.jp/get/{cid}"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    formula, charge = None, 0
    for line in r.text.splitlines():
        # FORMULA <spaces> <formula>
        m = re.match(r"^FORMULA\s+(.+)$", line)
        if m:
            formula = m.group(1).strip()
            continue
        # CHARGE <spaces> <int>
        m = re.match(r"^CHARGE\s+(-?\d+)$", line)
        if m:
            try:
                charge = int(m.group(1))
            except ValueError:
                pass
    return formula, charge

# -------- Equation parsing (robust to common arrows) --------
def normalize_arrow(eq):
    # normalize a few common arrows to "<=>"
    eq = re.sub(r"\s*(<[-=]+>|<=>|<->|⇌|↔|→|->|=>)\s*", " <=> ", eq)
    return eq

def parse_equation(equation):
    """
    Parse a reaction equation into substrates and products.
    Supports integer and fractional stoichiometric coefficients.
    Returns [(coeff, compound_id), ...] for each side.
    """
    equation = normalize_arrow(equation)
    if "<=>" not in equation:
        raise ValueError(f"No recognized arrow in equation: {equation}")
    left, right = equation.split("<=>")

    def parse_side(side):
        parts = [p.strip() for p in side.strip().split("+") if p.strip()]
        parsed = []
        for p in parts:
            # allow int or float coefficient
            m = re.match(r"^\s*([\d\.]+)\s+(\S+)\s*$", p)
            if m:
                coeff = float(m.group(1))
                cid = m.group(2)
            else:
                coeff, cid = 1.0, p.strip()
            parsed.append((coeff, cid))
        return parsed

    return parse_side(left), parse_side(right)

# -------- Formula parsing (only real elements, supports hydrates) --------
def parse_formula(formula):
    """
    Parse a chemical formula string into element counts.
    Supports dot/center-dot separated parts (e.g., 'C10H16N5O13P3·H2O').
    Ignores any stray characters outside [A-Za-z0-9·.].
    """
    if not formula:
        return Counter()

    counts = Counter()
    # split hydrate parts: dot or middle dot or whitespace clusters
    parts = re.split(r"[·\.]\s*|\s{2,}", formula.strip())
    for part in parts:
        if not part:
            continue
        # strip any non-alnum that might sneak in
        part = re.sub(r"[^A-Za-z0-9]", "", part)
        # scan symbol-by-symbol (1 or 2-letter symbols only)
        i = 0
        while i < len(part):
            if i+1 < len(part) and part[i:i+2] in ELEMENTS:
                sym = part[i:i+2]; i += 2
            elif part[i] in [s[:1] for s in ELEMENTS]:
                # take single letter; verify it's a valid 1-letter symbol
                sym1 = part[i]; i += 1
                # map to full symbol if exists (H, B, C, N, O, F, P, S, K, V, Y, I, W, U)
                if sym1 in ELEMENTS:
                    sym = sym1
                else:
                    # not a valid element (e.g., 'R' from 'FORMULA'); skip
                    continue
            else:
                # unknown letter chunk; skip it (guards against stray text)
                i += 1
                continue

            # optional number following the symbol
            j = i
            while j < len(part) and part[j].isdigit():
                j += 1
            num = int(part[i:j]) if j > i else 1
            counts[sym] += num
            i = j
    return counts

def multiply_formula(counts, coeff):
    return Counter({atom: n * coeff for atom, n in counts.items()})

# -------- Clean mapping: custom IDs -> KEGG IDs by token --------
def map_custom_to_kegg_equation(equation, mapping_df):
    custom2kegg = dict(zip(mapping_df["ID"], mapping_df["KEGG ID"]))
    subs, prods = parse_equation(equation)

    def map_side(pairs):
        mapped = []
        for coeff, cid in pairs:
            kid = custom2kegg.get(cid, cid)
            if pd.isna(kid) or kid is None:
                kid = cid  # leave unmapped if no KEGG id
            mapped.append((coeff, str(kid)))
        return mapped

    subs_k = map_side(subs)
    prods_k = map_side(prods)

    def side_to_str(pairs):
        parts = []
        for coeff, cid in pairs:
            if abs(coeff - 1.0) < 1e-12:  # display clean if coefficient is ~1
                parts.append(cid)
            else:
                parts.append(f"{coeff:g} {cid}")  # keep decimals clean
        return " + ".join(parts) if parts else "0"

    eq_kegg = f"{side_to_str(subs_k)} <=> {side_to_str(prods_k)}"
    return eq_kegg, subs_k, prods_k

# -------- Balance check with signed atom deltas --------
def check_balance(substrates, products, formula_map, charge_map, tol=1e-6):
    left_atoms, right_atoms = Counter(), Counter()
    left_charge, right_charge = 0.0, 0.0

    for coeff, cid in substrates:
        f = formula_map.get(cid)
        if f:
            left_atoms += multiply_formula(parse_formula(f), coeff)
        left_charge += charge_map.get(cid, 0) * coeff

    for coeff, cid in products:
        f = formula_map.get(cid)
        if f:
            right_atoms += multiply_formula(parse_formula(f), coeff)
        right_charge += charge_map.get(cid, 0) * coeff

    # compare atoms with tolerance
    all_atoms = set(left_atoms) | set(right_atoms)
    delta = {
        a: right_atoms.get(a, 0.0) - left_atoms.get(a, 0.0)
        for a in all_atoms
    }

    atoms_bal = all(abs(d) < tol for d in delta.values())
    charge_bal = abs(left_charge - right_charge) < tol

    # compact summary string for imbalances above tolerance
    imbalance_summary = ", ".join(
        [f"{'+' if d > 0 else ''}{d:g} {a}" for a, d in sorted(delta.items()) if abs(d) >= tol]
    ) or None

    return atoms_bal, charge_bal, left_atoms, right_atoms, left_charge, right_charge, delta, imbalance_summary

# -------- Pipeline: custom eq -> KEGG eq -> balance --------
def analyze_custom_equations(equations, mapping_df):
    rows = []
    for eq in equations:
        try:
            eq_kegg, subs_k, prods_k = map_custom_to_kegg_equation(eq, mapping_df)
        except Exception as e:
            rows.append({
                "equation_custom": eq,
                "equation_kegg": None,
                "atom_balanced": None,
                "charge_balanced": None,
                "left_charge": None,
                "right_charge": None,
                "atom_delta": None,
                "atom_imbalance": None,
                "notes": f"Parse error: {e}",
            })
            continue

        # collect all KEGG compound IDs in the mapped equation
        cids = [cid for _, cid in subs_k + prods_k]

        # fetch KEGG info once per unique cid
        uniq = list(dict.fromkeys(cids))  # preserve order, unique
        compound_info = {cid: get_compound_info(cid) for cid in uniq}
        formula_map = {cid: v[0] for cid, v in compound_info.items()}
        charge_map = {cid: v[1] for cid, v in compound_info.items()}

        atoms_bal, charge_bal, left_atoms, right_atoms, left_charge, right_charge, delta, imbalance_summary = \
            check_balance(subs_k, prods_k, formula_map, charge_map)

        notes = []
        if not atoms_bal:
            notes.append("Atoms not balanced")
        if not charge_bal:
            notes.append("Charge not balanced")
        if not notes:
            notes.append("OK")

        rows.append({
            "equation_custom": eq,
            "equation_kegg": eq_kegg,
            "atom_balanced": atoms_bal,
            "charge_balanced": charge_bal,
            "left_charge": left_charge,
            "right_charge": right_charge,
            "atom_delta": delta,                      # dict, e.g. {'H': -1, 'P': +1}
            "atom_imbalance": imbalance_summary,      # string, e.g. "+1 P, -1 H"
            "notes": "; ".join(notes),
        })

    return pd.DataFrame(rows)

def analyze_stoichiometry_matrix(
    stoich_df,
    mapping_df,
    formula_map=None,
    charge_map=None,
    tol=1e-6,
    only_unbalanced=False,
):
    """
    Analyze reactions from a stoichiometric matrix format.

    Parameters
    ----------
    stoich_df : pd.DataFrame
        Rows = metabolites (index like 'EXH2O', 'EXATP', etc., abbrev = index[2:]).
        Columns = reactions, values = stoichiometric coefficients.
        Negative = substrate, Positive = product.

    mapping_df : pd.DataFrame
        Must contain columns ["ID", "KEGG ID"].

    formula_map, charge_map : dict, optional
        Precomputed maps {kegg_id: formula_str} and {kegg_id: charge_int}.
        If not provided, compound info will be fetched via KEGG (with caching).

    tol : float, default 1e-6
        Tolerance for atom/charge balance checks.

    only_unbalanced : bool, default False
        If True, only return reactions that are not atom- and/or charge-balanced.

    Returns
    -------
    pd.DataFrame with one row per reaction, including balance check results.
    """
    rows = []
    abbrev2kegg = dict(zip(mapping_df["ID"], mapping_df["KEGG ID"]))

    for rxn in stoich_df.columns:
        coeffs = stoich_df[rxn].dropna()
        substrates, products = [], []

        for met_full, coeff in coeffs.items():
            if abs(coeff) < tol:
                continue
            # abbreviation from index[2:]
            abbrev = met_full[2:]
            kegg_id = abbrev2kegg.get(abbrev, abbrev)
            if pd.isna(kegg_id) or kegg_id is None:
                kegg_id = abbrev

            if coeff < 0:
                substrates.append((-coeff, str(kegg_id)))
            else:
                products.append((coeff, str(kegg_id)))

        # Build KEGG-style equation string
        def side_to_str(pairs):
            parts = []
            for coeff, cid in pairs:
                if abs(coeff - 1.0) < tol:
                    parts.append(cid)
                else:
                    parts.append(f"{coeff:g} {cid}")
            return " + ".join(parts) if parts else "0"

        eq_kegg = f"{side_to_str(substrates)} <=> {side_to_str(products)}"

        # Build maps if not precomputed
        if formula_map is None or charge_map is None:
            cids = [cid for _, cid in substrates + products]
            uniq = list(dict.fromkeys(cids))
            compound_info = {cid: get_compound_info(cid) for cid in uniq}
            f_map = {cid: v[0] for cid, v in compound_info.items()}
            c_map = {cid: v[1] for cid, v in compound_info.items()}
        else:
            f_map, c_map = formula_map, charge_map

        # Check balance
        atoms_bal, charge_bal, left_atoms, right_atoms, left_charge, right_charge, delta, imbalance_summary = \
            check_balance(substrates, products, f_map, c_map, tol=tol)

        # Skip balanced reactions if requested
        if only_unbalanced and atoms_bal and charge_bal:
            continue

        notes = []
        if not atoms_bal:
            notes.append("Atoms not balanced")
        if not charge_bal:
            notes.append("Charge not balanced")
        if not notes:
            notes.append("OK")

        rows.append({
            "reaction_id": rxn,
            "equation_kegg": eq_kegg,
            "atom_balanced": atoms_bal,
            "charge_balanced": charge_bal,
            "left_charge": left_charge,
            "right_charge": right_charge,
            "atom_delta": delta,
            "atom_imbalance": imbalance_summary,
            "notes": "; ".join(notes),
        })

    return pd.DataFrame(rows)


## Run Stoichiometric Balance Check

Apply the balance-checking pipeline to every reaction in the loaded dataset. Reaction equations are first translated from custom abbreviations to KEGG compound IDs, then checked for atom and charge balance. Results (including any imbalance details) are stored in `df_results`.

<div style="border-left:5px solid #2196F3; background:rgba(33,150,243,0.08); padding:10px 14px; margin:12px 0; border-radius:3px;">
<b>&#9998; TO DO:</b> Inspect <code>df_results</code> below. If any <b>non-exchange</b> reaction is imbalanced, edit your Excel file to fix it, then re-run from the top. H2O and H+; may be added to either side of a reaction to balance atoms and charge as indicated by the imbalance shown.
</div>


In [71]:
# equations written in your custom Abbreviations
reactions = list(df_reactions.loc[:, 'Reaction stoichiometry'])

# mapping table (df_metabolites) should have: ["KEGG ID", "ID"]
df_balance = analyze_custom_equations(reactions, df_metabolites)

## Build the COBRApy Model

Create a new COBRApy model (`Example1_EMPglycolysis`), add all metabolites from the spreadsheet, then add all reactions with their stoichiometry. Reactions marked as irreversible in the spreadsheet have their lower bound set to 0.

In [72]:

model = cobra.Model(name=MODEL_NAME)

cobra_mets = []
for i in df_metabolites.index:
  cobra_mets.append(cobra.Metabolite(id=df_metabolites.loc[i, 'ID'], name=df_metabolites.loc[i, 'Name'], compartment='c'))

model.add_metabolites(cobra_mets)


In [73]:
cobra_reacs = []

for i in df_reactions.index:

  r = cobra.Reaction(id=df_reactions.loc[i, 'ID'], name=df_reactions.loc[i, 'Name'], lower_bound=-1000, upper_bound=1000)

  model.add_reactions([r])

  r.reaction = df_reactions.loc[i, 'Reaction stoichiometry']

  if df_reactions.loc[i, 'Reversibility'] == 0:
    print(f"{r.id}: {r.reaction} is irreversible")
    r.lower_bound = 0
  cobra_reacs.append(r)

PTS: GLCex + PEP <=> G6P + PYR is irreversible
PGI: G6P <=> F6P is irreversible
PFK: ATP + F6P <=> ADP + FBP is irreversible
ALDO: FBP <=> DHAP + G3P is irreversible
TPI: DHAP <=> G3P is irreversible
GAPDH: G3P + NAD + Pi <=> BPG + H + NADH is irreversible
PGK: ADP + BPG <=> ATP + PG3 is irreversible
PGM: PG3 <=> PG2 is irreversible
ENO: PG2 <=> H2O + PEP is irreversible
PYK: ADP + PEP <=> ATP + PYR is irreversible
PFL: COA + PYR <=> ACCOA + FOR is irreversible
PTA: ACCOA + Pi <=> ACTP + COA is irreversible
ACK: ACTP + ADP <=> AC + ATP is irreversible
ACALD: ACCOA + H + NADH <=> ACALD + COA + NAD is irreversible
ADH: ACALD + H + NADH <=> ETOH + NAD is irreversible
TETOH: ETOH <=> ETOHex is irreversible
EXETOH: ETOHex <=>  is irreversible
TAC: AC <=> ACex is irreversible
EXAC: ACex <=>  is irreversible
TFOR: FOR <=> FORex is irreversible
EXFOR: FORex <=>  is irreversible
TATP: ATP <=> ATPex is irreversible
EXATP: ATPex <=>  is irreversible


## Configure Exchange Reactions

Define which metabolites act as substrate or as secretable products. Exchange reactions that are not in the substrate, product, or freely reversible lists are restricted or removed, so FBA operates under realistic constraints.

In [74]:
# Uses SUBSTRATES, PRODUCTS, and REV_ALLOWED defined in the inputs cell above.
# If reactions are removed here and you re-run this cell, first re-run all cells above to rebuild the model.
for r in model.reactions:
  if r.id[:2] == 'EX' and r.id not in REV_ALLOWED and r.id not in SUBSTRATES:
    r.lower_bound = 0

  if r.id[:2] == 'EX' and r.id not in PRODUCTS and r.id not in SUBSTRATES and r.id not in REV_ALLOWED:
    print(f'{r.id} was removed')
    model.remove_reactions([r])


## Sanity check: Maximise ATP Yield without substrate uptake

Run Flux Balance Analysis with ATP production as the objective and all substrate uptake blocked. If ATP production is nonzero, thermodynamically infeasible cycles are present in the model.

<div style="border-left:5px solid #2196F3; background:rgba(33,150,243,0.08); padding:10px 14px; margin:12px 0; border-radius:3px;">
<b>&#9998; TO DO:</b> If any nonzero fluxes are printed below, your model contains a cycle that produces energy without consuming substrate. Edit your Excel file (e.g. set the relevant reactions to irreversible) and re-run from the top.
</div>


In [75]:
model.objective = model.reactions.get_by_id(ENERGY_PRODUCT)

# Block uptake of every substrate and remember original lower bounds
original_lb = {}
for s in SUBSTRATES:
    r = model.reactions.get_by_id(s)
    original_lb[s] = r.lower_bound
    r.lower_bound = 0

result = model.optimize()
fluxes = result.fluxes

for f in fluxes.index:
  if abs(fluxes[f]) > 1e-6:
    print(f'{f}: {fluxes[f]}')

# Restore original lower bounds
for s, lb in original_lb.items():
    model.reactions.get_by_id(s).lower_bound = lb


## Sanity check: Maximise Individual Product Yields

For each product in `CARBON_PRODUCTS`, the maximum theoretical yield from the substrate is computed via FBA and printed below.

<div style="border-left:5px solid #2196F3; background:rgba(33,150,243,0.08); padding:10px 14px; margin:12px 0; border-radius:3px;">
<b>&#9998; TO DO:</b> If a product yield is zero or the optimisation is infeasible, either remove that product from <code>CARBON_PRODUCTS</code> in the inputs cell, or add the missing reactions to your Excel file and re-run.
</div>


In [76]:
model.reactions.get_by_id(SUBSTRATES[0]).bounds = -10, 1000

for p in CARBON_PRODUCTS:
  model.objective = model.reactions.get_by_id(p)
  result = model.optimize()
  print(f'product {p[2:]} : {result.objective_value}')

model.reactions.get_by_id(SUBSTRATES[0]).bounds = -1000, 1000


product ETOH : 10.0
product AC : 10.000000000000004
product FOR : 20.000000000000004


## Data Cleanup — Fix NaN / None Values prior to saving

Before exporting, scan the model for `NaN` or `None` values in reaction bounds, metabolite charges, formula weights, and reaction names, and replace them with safe defaults. This prevents serialisation errors when saving to JSON or SBML.

In [77]:

# Fix NaN in bounds
for r in model.reactions:
    if math.isnan(r.lower_bound):
        r.lower_bound = 0.0
        print(r.id)
    if math.isnan(r.upper_bound):
        r.upper_bound = 0.0
        print(r.id)

In [78]:
# Fix NaN in metabolite charges or formula weights
for m in model.metabolites:
    if m.charge is None or (isinstance(m.charge, float) and math.isnan(m.charge)):
        m.charge = 0
        print(m.id)
    if m.formula_weight is None or (isinstance(m.formula_weight, float) and math.isnan(m.formula_weight)):
        m.formula_weight = 0.0
        print(m.id)


GLCex
GLC
G6P
F6P
FBP
G3P
DHAP
BPG
PG3
PG2
PEP
PYR
FOR
ACCOA
ACTP
AC
ACALD
ETOH
ATP
ADP
Pi
NAD
NADH
H2O
H
COA
ETOHex
ACex
FORex
ATPex
ADPex
Piex
H2Oex
Hex


In [79]:

for r in model.reactions:
    if not isinstance(r.name, str) or r.name is None or (isinstance(r.name, float) and math.isnan(r.name)):
        print(r.id)
        r.name = r.id  # fallback to reaction ID

## Export the Model - Optional

Save the finished model in two interoperable formats: JSON (COBRApy-native, readable by Python tools) and SBML (standard exchange format compatible with COPASI, libSBML, Cameo, and others).

In [80]:
cobra.io.save_json_model(model, MODEL_NAME + '.json')
cobra.io.write_sbml_model(model, MODEL_NAME + '.xml')


## Find blocked reactions and remove

These *blocked* reactions cannot carry flux in any feasible steady state and are removed from the model before EFM enumeration, keeping the stoichiometric matrix as small as possible and speeding up the calculation.

In [81]:
# Find blocked reactions
blocked_reactions = cobra.flux_analysis.find_blocked_reactions(model)

# Remove these reactions from the model
for rxn_id in blocked_reactions:
    model.remove_reactions([model.reactions.get_by_id(rxn_id)])

# Print summary
print(f"Total reactions removed: {len(blocked_reactions)}")
print(f"Remaining reactions in the model: {len(model.reactions)}")

Total reactions removed: 2
Remaining reactions in the model: 30


## Build Stoichiometric Matrix and Reversibility Array

Extract the stoichiometric matrix **S** from the pruned COBRApy model and remove any rows that are entirely zero (metabolites no longer involved in any remaining reaction). A reversibility array is built in parallel: a reaction is marked reversible (`1`) if its lower bound is negative, and irreversible (`0`) otherwise. Both objects are the direct inputs to the EFM enumerator.

In [82]:
S = cobra.util.array.create_stoichiometric_matrix(model)

S_nonzero = S[~np.all(S == 0, axis=1)]
metabolite_ids = np.array([m.id for m in model.metabolites])[~np.all(S == 0, axis=1)]

S = S_nonzero

reversibility_array = []
for r in model.reactions:
  if r.id[:2] == 'EX' and r.id not in REV_ALLOWED and r.id not in SUBSTRATES:
    r.lower_bound = 0

  if r.lower_bound < 0:
    reversibility_array.append(1)
  else:
    reversibility_array.append(0)

print(S.shape)


(32, 30)


## Determine rank and dimensions of stoichiometric matrix

## Elementary Flux Mode (EFM) Enumeration

Enumerate all Elementary Flux Modes (EFMs) of the network using `efmtool`. An EFM is a minimal set of reactions that can sustain a non-zero steady-state flux while respecting thermodynamic reversibility constraints — every feasible metabolic flux distribution can be written as a non-negative linear combination of EFMs. The call is wrapped in a timer because calculation time scales super-exponentially with network size.

In [83]:
start = time.time()

efms = efmtool.calculate_efms(S, reversibility_array, [r.id for r in model.reactions], metabolite_ids)
end = time.time()

print(f'EFM calculation took {end-start} s')
print(f'Number of EFMs: {efms.shape[1]}')

df_efms = pd.DataFrame(efms, index=[r.id for r in model.reactions], columns=[f'EFM_{i+1}' for i in range(efms.shape[1])])

EFM calculation took 4.1580650806427 s
Number of EFMs: 1


In [84]:
df_efms

,EFM_1
PTS,1.0
PGI,1.0
PFK,1.0
ALDO,1.0
TPI,1.0
GAPDH,2.0
PGK,2.0
PGM,2.0
ENO,2.0
PYK,1.0


## Validate EFMs — Rank Check

Verify that every candidate EFM returned by `efmtool` satisfies the elementary-mode rank condition: for a valid EFM the number of active (non-zero) reactions must equal the rank of the corresponding sub-stoichiometric matrix plus one (`n_active - rank = 1`). Candidates that fail this check are collected in `not_efm` for further inspection.

In [85]:
not_efm = {}

for efm in df_efms.columns:
    # Take active (nonzero) indices in efm
    active_indices = np.where(df_efms[efm].values != 0)[0]
    # Take those columns from S
    S_active = S[:, active_indices]
    # Delete rows with only zeros from S
    nonzero_row_mask = ~(np.all(S_active == 0, axis=1))
    S_active_nz = S_active[nonzero_row_mask, :]
    # Calculate ncolumns - rank
    ncolumns = S_active_nz.shape[1]
    rank = np.linalg.matrix_rank(S_active_nz)

    if ncolumns-rank!=1:
        not_efm[efm] = ncolumns - rank

if len(not_efm) > 0:
    print('There are EFMs found that do not pass the rank check' )

## Normalise and Collapse EFMs

Normalise each EFM by the absolute glucose-uptake flux so all yields are expressed per unit of glucose consumed.

In [86]:
# Normalize each EFM by absolute substrate uptake flux
row_id = SUBSTRATES[0]
denominator = np.abs(df_efms.loc[row_id].values)
nonzero_mask = denominator != 0

df_normalized = df_efms.copy()
df_normalized.loc[:, nonzero_mask] = df_efms.loc[:, nonzero_mask] / denominator[nonzero_mask]


In [87]:
df_normalized

,EFM_1
PTS,1.0
PGI,1.0
PFK,1.0
ALDO,1.0
TPI,1.0
GAPDH,2.0
PGK,2.0
PGM,2.0
ENO,2.0
PYK,1.0


## Count Metabolic Reactions per EFM

To characterise how many enzymatic steps each EFM uses, we count the reactions that carry non-zero flux — but only genuine metabolic reactions. Two categories are excluded:

1. **Exchange reactions** (IDs starting with `EX`): these represent uptake/secretion across the model boundary and are not catalysed by enzymes.
2. **Pseudoreaction transports of H₂O, H⁺, ATP, ADP, and Pᵢ**: the transport of these currency/cofactor metabolites is obligatorily coupled to the chemistry already counted in the metabolic reactions (e.g. ATP hydrolysis couples to PFK). Counting them separately would double-count the same biochemical event.

The set of pseudo-transport reactions is identified automatically: any non-exchange reaction whose metabolite set is entirely contained within {H₂O, H⁺, ATP, ADP, Pᵢ} (and their extracellular counterparts) is labelled a pseudoreaction and excluded.

In [88]:
# PSEUDO_METS is defined in the User Inputs cell above

# Non-EX reactions whose entire metabolite set is within PSEUDO_METS
pseudo_transport_rxns = {
    r.id for r in model.reactions
    if not r.id.startswith('EX')
    and set(m.id for m in r.metabolites).issubset(PSEUDO_METS)
}

# Rows to keep: not an exchange, not a pseudo-transport, not a thermodynamic summary row
thermo_rows = {'dG0prime value', 'dG0prime error', 'dGm value', 'dGm error'}
metabolic_only = df_normalized[
    ~df_normalized.index.str.startswith('EX') &
    ~df_normalized.index.isin(pseudo_transport_rxns) &
    ~df_normalized.index.isin(thermo_rows)
]



# Count reactions with nonzero flux per EFM
rxn_counts = (metabolic_only != 0).sum().rename('active_metabolic_reactions')

df_normalized.loc['n_metabolic_reactions'] = pd.Series(dtype=object)
for efm in df_normalized.columns:
    df_normalized.loc['n_metabolic_reactions', efm] = rxn_counts[efm]

print("Excluded pseudo-transport reactions:", sorted(pseudo_transport_rxns))
print()
print("Metabolic reaction count per EFM:")
print(rxn_counts.to_frame().T)

Excluded pseudo-transport reactions: ['TADP', 'TATP', 'TH2O', 'TransPi']

Metabolic reaction count per EFM:
                            EFM_1
active_metabolic_reactions     18


## Compute Overall Reaction Thermodynamics via eQuilibrator

For each EFM, the net stoichiometry is read from the exchange-flux rows and translated into a KEGG-formatted reaction string. eQuilibrator is then queried for two thermodynamic quantities:

- **ΔG°′** (`dG0prime`): standard Gibbs free energy change at pH 7 and ionic strength 0.25 M.
- **ΔGm′** (`dGm`): physiological Gibbs free energy change at millimolar metabolite concentrations.

Both the central value and the propagated uncertainty are stored as additional rows in `df_normalized`.

In [89]:
metabolite_map = dict(zip(df_metabolites.loc[:, 'ID'], df_metabolites.loc[:, 'KEGG ID']))

stoich_df = df_normalized.loc[df_normalized.index.str.startswith('EX')]
uniq_cids = df_metabolites["KEGG ID"].dropna().unique()
compound_info = {cid: get_compound_info(cid) for cid in uniq_cids}
formula_map = {cid: v[0] for cid, v in compound_info.items()}
charge_map  = {cid: v[1] for cid, v in compound_info.items()}
efm_balance = analyze_stoichiometry_matrix(stoich_df, df_metabolites, formula_map, charge_map, only_unbalanced=True)

df_normalized.loc['dG0prime value'] = pd.Series(dtype=object)
df_normalized.loc['dG0prime error'] = pd.Series(dtype=object)
df_normalized.loc['dGm value'] = pd.Series(dtype=object)
df_normalized.loc['dGm error'] = pd.Series(dtype=object)


for reac in df_normalized.columns:
  series = df_normalized.loc[df_normalized.index.str.startswith('EX'), reac]
  reac_dict = series[series != 0].to_dict()

  kegg_dict = {}
  for k, v in reac_dict.items():
    metab = k[2:]# For this it is important that all the names of the exchanges are exactly EX{METAB ABBREVIATION}

    kegg_code = metabolite_map[metab]

    kegg_dict[kegg_code] = v

  kegg_string = dict_to_kegg_reaction(kegg_dict)
  overall_reaction = cc.parse_reaction_formula(kegg_string)


  dG0_prime = cc.standard_dg_prime(overall_reaction)
  dGm = cc.physiological_dg_prime(overall_reaction)


  df_normalized.loc['dG0prime value', reac] = dG0_prime.value.magnitude
  df_normalized.loc['dG0prime error', reac] = dG0_prime.error.magnitude
  df_normalized.loc['dGm value', reac] = dGm.value.magnitude
  df_normalized.loc['dGm error', reac] = dGm.error.magnitude


df_normalized.loc['dG0prime value/nReactions'] = df_normalized.loc['dG0prime value'] / df_normalized.loc['n_metabolic_reactions']
df_normalized.loc['dGm value/nReactions'] = df_normalized.loc['dGm value'] / df_normalized.loc['n_metabolic_reactions']


df_normalized

,EFM_1
PTS,1.000000
PGI,1.000000
PFK,1.000000
ALDO,1.000000
TPI,1.000000
GAPDH,2.000000
PGK,2.000000
PGM,2.000000
ENO,2.000000
PYK,1.000000
